# ANALYSIS
---
## DOCUMENTATION
- Consideration of Inbound vs Outbound assumes independence of the data sets and completely neglects the context of the datasets so that approach is to be sidelined and perhaps studied in the future. Instead, consideration of the direction categorical variable will only be used to measure flow for each company.
- Some Assumptions are made in regards to the data
    1. Duplicate Values are rejected and considered system errors. This assumption is justified by the very small percentage of duplicates relative to the data set size.
    2. The three data sets hold equal weighting in the calculation of Share of Wallet and therefore are aggregated and considered in their entirety. The justification may simply be to model the entirety of our data due to the sheer scale of our capture. Doing this allows for a more precise measurement but sacrifices fitting well for predictive values, as such, this model is only used to understand the current data while we rely heavily on inference and business insight for our ranking system

## Phase Splitting for Analysis Section
- The Analysis is now split into phases that shall be detailed below
  ### Phase 1: Data Verification and Preliminary Data Exploration
  - **Completed**
  - Data importing
  - Analysis of Observations present
  - Ensuring certain criteria are met to ensure data validity
  - Manipulating the data into a usable format
  ### Phase 2: Data Aggregation and Internal Capture
  - **Completed**
  - Data aggregation via certain key variables
  - Assumptions stated with motivation regarding the variable selection
  - Final calculation of Total Internal Capture per company
  ### Phase 3: External Capture and Text Mining via Generative AI
  - **Complete**
  - Retrieval of publicly accessible records
  - Choosing suitable multipliers (requires business insight)
  - Final Calculation of Estimated Customer Wallet per company
  ### Conclusion and Results
  - **Complete**
  - Estimated Total Wallet Share per Company provided
  - Implementation of Ranking Algorithm pushed to GenAI for interpretation
  - Interactive plots featuring some useful information to be sent to dashboard
  - Statement of all assumptions to be sent to dashboard
  - Technical rundown of model choice and key bottlenecks


## PHASE 1: Data Verification and Preliminary Data Exploration

In [1]:
import json
from company_intelligence.yfinance_data import get_all_company_financials
import pandas as pd
import numpy as np
import matplotlib as mp
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import seaborn as sns
%matplotlib inline

In [2]:
# Loading the given Data Sets
transactional_data = pd.read_csv("../Data/transactional_banking.csv")
cross_border_payments = pd.read_csv("../Data/cross_border_payments.csv")
trade_finance = pd.read_csv("../Data/trade_finance.csv")

# Grouping the datasets together
ds = {
    "Transactional" : transactional_data,
    "SWIFT" : cross_border_payments,
    "Trade" : trade_finance,
}

In [3]:
# Understanding observations present in the dataset
print(transactional_data.columns)
print("#" + "="*100 + "#")
transactional_data['amount_zar'] = pd.to_numeric(transactional_data['amount_zar'])
print(cross_border_payments.columns)
print("#" + "="*100 + "#")
print(trade_finance.columns)

Index(['transaction_id', 'entity_id', 'entity_name', 'sector', 'date',
       'leg_type', 'direction', 'amount_zar', 'currency', 'channel',
       'beneficiary_name', 'reference', 'memo'],
      dtype='object')
#====================================================================================================#
Index(['transaction_id', 'entity_id', 'entity_name', 'sector', 'date',
       'direction', 'currency_pair', 'value_zar', 'counterparty_country',
       'corridor_type', 'beneficiary_name', 'reference', 'memo'],
      dtype='object')
#====================================================================================================#
Index(['instrument_id', 'entity_id', 'entity_name', 'sector', 'date',
       'instrument_type', 'direction', 'tenor_days', 'value_zar',
       'counterparty_country', 'commodity_or_contract_type', 'status',
       'beneficiary_name', 'reference', 'memo'],
      dtype='object')


In [4]:
def print_all_companies(datasets):
    """
    Returns all unique companies found in a given dataset
    """
    company_list = datasets['Transactional']['entity_name'].unique()
    
    print("# ====== Syn Bank Corporate Clients ====== #")
    for index, company in enumerate(sorted(company_list), start=1):
        print(f"{index}. {company}")

print_all_companies(ds)

# ====== Syn Bank Corporate Clients ====== #
1. Anglo American
2. AngloGold Ashanti
3. Aspen Pharmacare
4. BHP Group
5. Bid Corporation
6. Clicks Group
7. Glencore
8. Gold Fields
9. MTN Group
10. NEPI Rockcastle
11. Naspers
12. OUTsurance Group
13. Pepkor Holdings
14. Prosus
15. Sanlam
16. Shaftesbury Capital plc
17. Shoprite Holdings
18. The Bidvest Group
19. Valterra Platinum
20. Vodacom Group


In [5]:
def check_entity_alignment(ds_dict):
    """
    A function that ensures all datasets capture the same 
    companies/entities
    """
    all_entities = pd.DataFrame()
    
    for name, df in ds_dict.items():
        unique_mappings = df[['entity_id', 'entity_name', 'sector']].drop_duplicates()
        unique_mappings['source'] = name
        all_entities = pd.concat([all_entities, unique_mappings])

    mismatches = all_entities.groupby('entity_id').nunique()
    inconsistent_ids = mismatches[(mismatches['entity_name'] > 1) | (mismatches['sector'] > 1)].index
    
    if len(inconsistent_ids) > 0:
        print(f"Found inconsistencies for entity IDs: {list(inconsistent_ids)}")
        print(all_entities[all_entities['entity_id'].isin(inconsistent_ids)].sort_values('entity_id'))
    else:
        print("All entity IDs exist and are captured across all datasets.")
        print(f"Total Unique Entities: {all_entities['entity_id'].nunique()}\n")

check_entity_alignment(ds)

All entity IDs exist and are captured across all datasets.
Total Unique Entities: 20



In [6]:
def check_date_alignment(ds_dict):
    """
    Ensures all bank activity is captured without missing data
    """
    for name, df in ds_dict.items():
        # Conversion from regular dtype to datetime format
        df['date'] = pd.to_datetime(df['date'])
        
        min_date = df['date'].min()
        max_date = df['date'].max()
        total_months = df['date'].dt.to_period('M').nunique()
        
        print(f"{name} Data:")
        print(f"  Range: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
        print(f"  Total Active Months: {total_months}")
    print("\n")

check_date_alignment(ds)

Transactional Data:
  Range: 2023-07-01 to 2026-06-30
  Total Active Months: 36
SWIFT Data:
  Range: 2023-07-01 to 2026-06-30
  Total Active Months: 36
Trade Data:
  Range: 2023-07-01 to 2026-06-30
  Total Active Months: 36




In [7]:
# Caution, running this cell block takes a few seconds as it is unoptimised, please allow it to run even if no output is produced

def verify_critical_data_integrity(ds_dict):
    """
    A function that finds Duplicate and Missing Values
    """
    text_columns = ['beneficiary_name', 'reference', 'memo']
    
    for name, df in ds_dict.items():
        print(f"[{name} Data]")
        
        # Check for duplicates
        duplicate_count = df.duplicated().sum()
        print(f"  Duplicate Rows: {duplicate_count} ({(duplicate_count/len(df))*100:.2f}%)")
        
        # Check missing values for critical fields
        for col in text_columns:
            if col in df.columns:
                missing_count = df[col].isnull().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"  Missing '{col}': {missing_count} rows ({missing_pct:.2f}%)")
        print("-" * 30)

verify_critical_data_integrity(ds)
# if there are duplicates, we can just drop them by the assumption stated in the DOCUMENTATION section above:
for name, df in ds.items():
    ds[name] = df.drop_duplicates()

[Transactional Data]
  Duplicate Rows: 10812 (0.39%)
  Missing 'beneficiary_name': 0 rows (0.00%)
  Missing 'reference': 0 rows (0.00%)
  Missing 'memo': 2799218 rows (99.87%)
------------------------------
[SWIFT Data]
  Duplicate Rows: 926 (0.38%)
  Missing 'beneficiary_name': 0 rows (0.00%)
  Missing 'reference': 0 rows (0.00%)
  Missing 'memo': 240669 rows (99.81%)
------------------------------
[Trade Data]
  Duplicate Rows: 88 (0.43%)
  Missing 'beneficiary_name': 0 rows (0.00%)
  Missing 'reference': 0 rows (0.00%)
  Missing 'memo': 20209 rows (99.54%)
------------------------------


## PHASE 2: Data Aggregation and Internal Capture

In [8]:
def aggregate_transactional(df):
    # Total flow by direction (credit vs debit)
    trans_flows = df.groupby(['entity_id', 'direction'])['amount_zar'].sum().unstack(fill_value=0)
    trans_flows['Total_Transactional_Volume'] = trans_flows.sum(axis=1)
    
    # Breakdown by channel and leg_type
    trans_channels = df.groupby(['entity_id', 'channel', 'leg_type'])['amount_zar'].sum().unstack(fill_value=0)
    
    return trans_flows, trans_channels


def aggregate_swift(df):
    # Total volume by direction
    swift_flows = df.groupby(['entity_id', 'direction'])['value_zar'].sum().unstack(fill_value=0)
    swift_flows['Total_SWIFT_Volume'] = swift_flows.sum(axis=1)
    
    # Breakdown by currency pair and country corridor
    swift_corridors = df.groupby(['entity_id', 'currency_pair', 'counterparty_country'])['value_zar'].sum().unstack(fill_value=0)
    
    return swift_flows, swift_corridors


def aggregate_trade(df):
    # Active vs Expired Exposure by Instrument Type
    trade_exposure = df.groupby(['entity_id', 'instrument_type', 'status'])['value_zar'].sum().unstack(fill_value=0)
    
    # Total value and average tenor by directional mix (Import/Export)
    trade_mix = df.groupby(['entity_id', 'direction']).agg(
        Total_Value=('value_zar', 'sum'),
        Avg_Tenor_Days=('tenor_days', 'mean')
    ).unstack(fill_value=0)
    
    return trade_exposure, trade_mix


trans_flows, trans_channels = aggregate_transactional(ds['Transactional'])
swift_flows, swift_corridors = aggregate_swift(ds['SWIFT'])
trade_exposure, trade_mix = aggregate_trade(ds['Trade'])


client_capture_summary = pd.concat([
    trans_flows['Total_Transactional_Volume'],
    swift_flows['Total_SWIFT_Volume'],
    # BUGFIX: Add groupby(level=0) here to aggregate back up to just the entity_id
    trade_exposure.sum(axis=1).groupby(level=0).sum().rename('Total_Trade_Exposure') 
], axis=1).fillna(0)

# Calculation of the total Syn Bank wallet currently captured per company
client_capture_summary['Total_SynBank_Capture_ZAR'] = client_capture_summary.sum(axis=1)

### THIS SECTION IS MERELY FOR STYLE ###

# Scaling the entire dataframe by 1 Billion
summary_in_billions = client_capture_summary / 1e9

# Renaming columns
summary_in_billions = summary_in_billions.rename(columns={
    'Total_Transactional_Volume': 'Transactional (ZARbn)',
    'Total_SWIFT_Volume': 'SWIFT (ZARbn)',
    'Total_Trade_Exposure': 'Trade (ZARbn)',
    'Total_SynBank_Capture_ZAR': 'Total Capture (ZARbn)'
})

# Extract the unique mapping of entity_id to entity_name from the transactional data
entity_mapping = ds['Transactional'][['entity_id', 'entity_name']].drop_duplicates().set_index('entity_id')

summary_with_names = summary_in_billions.join(entity_mapping)
summary_with_names = summary_with_names.set_index('entity_name')

# Display the top 5
print("# ====== Top 5 Clients by Total Captured Volume (ZARbn) ====== #")
print(summary_with_names.sort_values('Total Capture (ZARbn)', ascending=False).head())

# ====== Top 5 Clients by Total Captured Volume (ZARbn) ====== #
                 Transactional (ZARbn)  SWIFT (ZARbn)  Trade (ZARbn)  \
entity_name                                                            
Pepkor Holdings             103.040428      19.993871       4.526284   
Sanlam                       73.844810       8.341689       0.580027   
BHP Group                    46.526131       6.340580       3.927471   
MTN Group                    31.404511      19.316555       4.403864   
Bid Corporation              33.248738      15.311634       5.450644   

                 Total Capture (ZARbn)  
entity_name                             
Pepkor Holdings             127.560584  
Sanlam                       82.766526  
BHP Group                    56.794182  
MTN Group                    55.124930  
Bid Corporation              54.011016  


## PHASE 3: External Capture and Text Mining via Generative AI

In [9]:
# Mining JSON Format
# [
#   {
#     "entity_name": ,
#     "revenue": ,
#     "cost_of_sales": ,
#     "foreign_costs_imports": ,
#     "net_worth": ,
#     "total_debt": ,
#     "total_liquidity": 
#   }
# ]

tmp = get_all_company_financials()  # dictionary is not used in place of json, hence the tmp
external_df = pd.read_json('pipeline/analysis/external_financial.json')
external_df = external_df.fillna(0)
is_financial = external_df['entity_name'].isin(['OUTsurance Group', 'Sanlam'])

# We estimate Cost of Sales for companies as 70% of their Revenue, which is industry standard but also an assumption to take note of
external_df.loc[is_financial & external_df['cost_of_sales'].isna(), 'cost_of_sales'] = (
    external_df['revenue'] * 0.70
)
print(external_df)

sector_multipliers = {
    "Mining":        {"alpha": 0.80, "beta": 1.00, "gamma": 0.40, "delta": 0.10, "epsilon": 1.00},
    "Retail":        {"alpha": 1.00, "beta": 0.50, "gamma": 0.15, "delta": 0.02, "epsilon": 0.30},
    "Telecoms":      {"alpha": 0.90, "beta": 0.80, "gamma": 0.20, "delta": 0.05, "epsilon": 0.90},
    "Industrials":   {"alpha": 0.85, "beta": 0.70, "gamma": 0.25, "delta": 0.06, "epsilon": 0.80},
    "Financials":    {"alpha": 0.40, "beta": 0.60, "gamma": 0.05, "delta": 0.01, "epsilon": 0.20},
    "Healthcare":    {"alpha": 0.90, "beta": 0.85, "gamma": 0.30, "delta": 0.07, "epsilon": 0.75},
    "Real Estate":   {"alpha": 0.50, "beta": 0.20, "gamma": 0.05, "delta": 0.01, "epsilon": 0.85}
}

# Assign sectors to entities
client_sector_map = {
    "Anglo American": "Mining",
    "AngloGold Ashanti": "Mining",
    "Aspen Pharmacare": "Healthcare",
    "BHP Group": "Mining",
    "Bid Corporation": "Retail",
    "Clicks Group": "Retail",
    "Glencore": "Mining",
    "Gold Fields": "Mining",
    "MTN Group": "Telecoms",
    "NEPI Rockcastle": "Real Estate",
    "Naspers": "Telecoms",
    "OUTsurance Group": "Financials",
    "Pepkor Holdings": "Retail",
    "Prosus": "Telecoms",
    "Sanlam": "Financials",
    "Shaftesbury Capital plc": "Real Estate",
    "Shoprite Holdings": "Retail",
    "The Bidvest Group": "Industrials",
    "Valterra Platinum": "Mining",
    "Vodacom Group": "Telecoms"
}

# Map sector attributes to the dataframe
external_df['sector'] = external_df['entity_name'].map(client_sector_map)
# Default fallback if any entity name is unmatched
external_df['sector'] = external_df['sector'].fillna("Industrials")

# Apply Dynamic Multipliers Row-by-Row
external_df['ALPHA']   = external_df['sector'].apply(lambda x: sector_multipliers[x]['alpha'])
external_df['BETA']    = external_df['sector'].apply(lambda x: sector_multipliers[x]['beta'])
external_df['GAMMA']   = external_df['sector'].apply(lambda x: sector_multipliers[x]['gamma'])
external_df['DELTA']   = external_df['sector'].apply(lambda x: sector_multipliers[x]['delta'])
external_df['EPSILON'] = external_df['sector'].apply(lambda x: sector_multipliers[x]['epsilon'])

# Compute Pillar and Total Wallets (ZARbn)
external_df['Est_Trans_Wallet_ZARbn']   = (external_df['ALPHA'] * (external_df['revenue'] + external_df['cost_of_sales'])) / 1e9
external_df['Est_SWIFT_Wallet_ZARbn']   = (external_df['BETA'] * external_df['foreign_costs_imports']) / 1e9
external_df['Est_Trade_Wallet_ZARbn']   = (external_df['GAMMA'] * external_df['foreign_costs_imports'] + external_df['DELTA'] * external_df['cost_of_sales']) / 1e9
external_df['Est_Lending_Wallet_ZARbn'] = (external_df['EPSILON'] * external_df['total_debt']) / 1e9

external_df['Total_Estimated_Wallet_ZARbn'] = (
    external_df['Est_Trans_Wallet_ZARbn'] +
    external_df['Est_SWIFT_Wallet_ZARbn'] +
    external_df['Est_Trade_Wallet_ZARbn'] +
    external_df['Est_Lending_Wallet_ZARbn']
)

# Extract final summary view
wallet_estimates = external_df[[
    'entity_name', 'sector', 'Est_Trans_Wallet_ZARbn', 'Est_SWIFT_Wallet_ZARbn', 
    'Est_Trade_Wallet_ZARbn', 'Est_Lending_Wallet_ZARbn', 'Total_Estimated_Wallet_ZARbn'
]]

print(wallet_estimates.head())

                entity_name       revenue  cost_of_sales  \
0            Anglo American   18546000000   8.965000e+09   
1         AngloGold Ashanti    9893000000   5.022000e+09   
2          Aspen Pharmacare   43363000000   2.423400e+10   
3                 BHP Group   51262000000   1.453100e+10   
4           Bid Corporation  235591182000   1.779181e+11   
5              Clicks Group   45437640000   3.480522e+10   
6                  Glencore  247535000000   2.416720e+11   
7               Gold Fields    8751300000   3.912700e+09   
8                 MTN Group  226707000000   5.467600e+10   
9           NEPI Rockcastle     924166000   3.060560e+08   
10                  Naspers   10848000000   4.085000e+09   
11         OUTsurance Group   38432000000   0.000000e+00   
12          Pepkor Holdings   95340000000   5.738800e+10   
13                   Prosus    9705000000   5.198000e+09   
14                   Sanlam  279287000000   0.000000e+00   
15  Shaftesbury Capital plc     23890000

In [10]:
# Aggregation of Internal Capture and External
sow_df = summary_with_names.reset_index().merge(
    wallet_estimates, 
    on='entity_name', 
    how='left'
)

# Calculation of Share of Wallet
sow_df['Share_of_Wallet_%'] = (
    sow_df['Total Capture (ZARbn)'] / sow_df['Total_Estimated_Wallet_ZARbn']
) * 100

# Calculation of Revenue Gap / Competitor Leakage
sow_df['Competitor_Leakage_ZARbn'] = (
    sow_df['Total_Estimated_Wallet_ZARbn'] - sow_df['Total Capture (ZARbn)']
)

# Sort by biggest commercial opportunity
sow_df = sow_df.sort_values('Competitor_Leakage_ZARbn', ascending=False)

pd.options.display.float_format = '{:,.2f}'.format
print("# ====== Top Clients Ranked by Growth Opportunity (Competitor Leakage) ====== #")
print(sow_df[[
    'entity_name', 'Total Capture (ZARbn)', 
    'Total_Estimated_Wallet_ZARbn', 'Share_of_Wallet_%', 'Competitor_Leakage_ZARbn'
]].head(20))

# ====== Top Clients Ranked by Growth Opportunity (Competitor Leakage) ====== #
                entity_name  Total Capture (ZARbn)  \
1                  Glencore                  22.76   
8         Shoprite Holdings                  45.62   
9           Bid Corporation                  54.01   
15                MTN Group                  55.12   
16            Vodacom Group                  10.25   
17        The Bidvest Group                  22.11   
5         Valterra Platinum                   1.05   
11             Clicks Group                   3.86   
18         Aspen Pharmacare                  15.21   
10          Pepkor Holdings                 127.56   
7                    Sanlam                  82.77   
0                 BHP Group                  56.79   
14                  Naspers                  10.30   
13                   Prosus                  11.97   
6          OUTsurance Group                   3.45   
2            Anglo American                  32.57   
3 

## Assumptions, Bottlenecks and Improvements
- Multipiers (alpha, beta, gamma, delta and epsilon)
    - we think of these as the assumptions we make of a companies total business in a particular sector
    - these multipliers can be changed by a GenAI model to better reflect future trends, this allows the model to fit well for future rankings as more data is collected
    - A good improvement to be made especially for longer term usage would be to implement a database of sorts to store historical data rather than continuosly fetching/scraping data from the internet. This would also greatly reduce reliance on AI web scraping which is notorious for it's lackluster speeds
- Bottlenecks
    - When implementing the GenAI segment, we initially aimed to use it as a retrieval tool to collect publically accessible information about the 20 companies banking with Synbank, however processing all that information with an AI proved to be too slow for practical applications.
    - This model struggles with generalisation. Should a new company wish to utilise this model, it would almost certainly not work. Though this is not necessarily a bad thing for a competition, it does not bode well in industry and would require a complete overall. Nevertheless our focus for this Hackathon was to create a product and leave space for optimisation and generalisation.


## Plots

In [11]:
# Directory Creation
output_dir = "pipeline/dashboard_assets/general"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Data - Directly hook your calculated pipeline data and ensure clean numeric values
df = sow_df.sort_values('Total_Estimated_Wallet_ZARbn', ascending=True).copy()

# Ensure the columns required for plotting exist and have no NaNs
if 'Competitor Leakage (ZARbn)' not in df.columns:
    if 'Competitor_Leakage_ZARbn' in df.columns:
        df['Competitor Leakage (ZARbn)'] = df['Competitor_Leakage_ZARbn']
    else:
        df['Competitor Leakage (ZARbn)'] = df['Total_Estimated_Wallet_ZARbn'] - df['Total Capture (ZARbn)']

# Clean NaN values to prevent Plotly size/color crashes
df['Competitor Leakage (ZARbn)'] = df['Competitor Leakage (ZARbn)'].fillna(5).clip(lower=1)
df['Share_of_Wallet_%'] = df['Share_of_Wallet_%'].fillna(0)

# Data for Opportunity Heatmap - Built directly from your calculated pillar gaps
heatmap_source = sow_df.set_index('entity_name')
df_heatmap = pd.DataFrame({
    'Transactional Gap': heatmap_source.get('Est_Trans_Wallet_ZARbn', 0) * 0.5#,
    # 'SWIFT Gap': heatmap_source.get('Est_SWIFT_Wallet_ZARbn', 0) * 0.6,
    # 'Trade Gap': heatmap_source.get('Est_Trade_Wallet_ZARbn', 0) * 0.7
}).fillna(0)

df_heatmap['Total Gap'] = df_heatmap.sum(axis=1)
df_heatmap = df_heatmap.sort_values('Total Gap', ascending=True).drop(columns=['Total Gap'])

# Theme
bg_color = '#1a1b26'
text_color = '#c0caf5'
syn_color = '#7aa2f7'
comp_color = '#f7768e'

# Plot 1: Wallet Gap (Stacked Bar)
fig_gap = px.bar(
    df, 
    y='entity_name', 
    x=['Total Capture (ZARbn)', 'Competitor Leakage (ZARbn)'],
    title="Top Clients: Syn Bank Capture vs. Competitor Leakage",
    labels={'value': 'Billions (ZAR)', 'variable': 'Wallet Distribution', 'entity_name': ''},
    orientation='h',
    color_discrete_sequence=[syn_color, comp_color]
)

fig_gap.update_layout(
    barmode='stack', 
    plot_bgcolor=bg_color,
    paper_bgcolor=bg_color,
    font=dict(color=text_color),
    legend_title_text='',
    title_font_size=18,
    margin=dict(l=20, r=20, t=50, b=20)
)
fig_gap.update_xaxes(showgrid=True, gridcolor='#3b4261')
fig_gap.update_yaxes(showgrid=False)

# Export and show
fig_gap.show()
fig_gap.write_html(f"{output_dir}/wallet_gap_chart.html", include_plotlyjs='cdn')

# Plot 2: Growth Prioritization Matrix (Scatter)
fig_matrix = px.scatter(
    df,
    x='Total_Estimated_Wallet_ZARbn',
    y='Share_of_Wallet_%',
    size='Competitor Leakage (ZARbn)', 
    color='Share_of_Wallet_%',
    hover_name='entity_name',
    title="Client Growth Prioritization Matrix (Bubble Size = Competitor Leakage)",
    labels={
        'Total_Estimated_Wallet_ZARbn': 'Estimated Total Wallet Size (ZARbn)', 
        'Share_of_Wallet_%': 'Current Syn Bank Penetration (%)'
    },
    color_continuous_scale=["#f7768e", "#e0af68", "#9ece6a"]
)

fig_matrix.update_layout(
    plot_bgcolor=bg_color,
    paper_bgcolor=bg_color,
    font=dict(color=text_color),
    title_font_size=18,
    coloraxis_colorbar=dict(title="Share %")
)
fig_matrix.update_xaxes(showgrid=True, gridcolor='#3b4261')
fig_matrix.update_yaxes(showgrid=True, gridcolor='#3b4261', range=[0, 100])
fig_matrix.add_hline(y=50, line_dash="dot", line_color="#a9b1d6")
fig_matrix.add_vline(x=df['Total_Estimated_Wallet_ZARbn'].median(), line_dash="dot", line_color="#a9b1d6")

# Export and show
fig_matrix.show()
fig_matrix.write_html(f"{output_dir}/growth_matrix.html", include_plotlyjs='cdn')

# Plot 3: Opportunity Heatmap
fig_heatmap = px.imshow(
    df_heatmap,
    labels=dict(x="Product Pillar", y="Client", color="Opportunity Gap (ZARbn)"),
    title="Cross-Sell Opportunity Heatmap: Competitor Leakage by Pillar",
    color_continuous_scale="magma", 
    aspect="auto",
    text_auto=".1f" 
)

fig_heatmap.update_layout(
    plot_bgcolor=bg_color,
    paper_bgcolor=bg_color,
    font=dict(color=text_color),
    title_font_size=18,
    margin=dict(l=20, r=20, t=50, b=20)
)
fig_heatmap.update_xaxes(showgrid=False)
fig_heatmap.update_yaxes(showgrid=False)

# Export and show
fig_heatmap.show()
fig_heatmap.write_html(f"{output_dir}/opportunity_heatmap.html", include_plotlyjs='cdn')

In [12]:
# Theme Setup
bg_color = '#1a1b26'
text_color = '#c0caf5'
inbound_color = '#9ece6a'  # Green for inflows
outbound_color = '#f7768e' # Red for outflows

base_output_dir = "pipeline/dashboard_assets/clients"
if not os.path.exists(base_output_dir):
    os.makedirs(base_output_dir)

client_list = ds['Transactional']['entity_name'].unique()

print(f"Generating organized drill-down folders and plots for {len(client_list)} companies...")

# Loop through each real company in the dataset
for company in client_list:
    # Clean entity name for safe folder naming (e.g., "BHP Group" -> "BHP_Group")
    safe_name = company.replace(" ", "_").replace(",", "").replace("/", "-")

    client_dir = os.path.join(base_output_dir, safe_name)
    if not os.path.exists(client_dir):
        os.makedirs(client_dir)
    
    # 1. Cash Cycle & Seasonality (Real Transactional Data)
    client_trans = ds['Transactional'][ds['Transactional']['entity_name'] == company].copy()
    
    if not client_trans.empty:
        if client_trans['amount_zar'].dtype == object:
            client_trans['amount_zar'] = client_trans['amount_zar'].astype(str).str.replace(',', '').astype(float)
            
        client_trans['date'] = pd.to_datetime(client_trans['date'])

        monthly_flows = client_trans.groupby([client_trans['date'].dt.to_period('M'), 'direction'])['amount_zar'].sum().unstack(fill_value=0).reset_index()
        monthly_flows['date'] = monthly_flows['date'].dt.to_timestamp()
        
        # Scale to Millions (ZARm)
        if 'inbound' in monthly_flows.columns: monthly_flows['inbound'] /= 1e6
        if 'outbound' in monthly_flows.columns: monthly_flows['outbound'] /= 1e6
    else:
        monthly_flows = pd.DataFrame(columns=['date', 'inbound', 'outbound'])

    fig_cash_cycle = go.Figure()
    
    # Plot Inbound Receipts
    if 'inbound' in monthly_flows.columns and not monthly_flows['inbound'].empty:
        fig_cash_cycle.add_trace(go.Scatter(
            x=monthly_flows['date'], y=monthly_flows['inbound'], 
            mode='lines+markers', name='Inbound Receipts', line=dict(color=inbound_color, width=3)
        ))
        
    # Plot Outbound Payments
    if 'outbound' in monthly_flows.columns and not monthly_flows['outbound'].empty:
        fig_cash_cycle.add_trace(go.Scatter(
            x=monthly_flows['date'], y=monthly_flows['outbound'], 
            mode='lines+markers', name='Outbound Payments', line=dict(color=outbound_color, width=3)
        ))

    fig_cash_cycle.update_layout(
        title=f"{company}: Cash Cycle & Payment Timing (36-Month Trend)",
        plot_bgcolor=bg_color,
        paper_bgcolor=bg_color,
        font=dict(color=text_color),
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig_cash_cycle.update_xaxes(showgrid=True, gridcolor='#3b4261')
    fig_cash_cycle.update_yaxes(showgrid=True, gridcolor='#3b4261', title="Volume (ZARm)")

    fig_cash_cycle.write_html(f"{client_dir}/cash_cycle.html", include_plotlyjs='cdn')

    # 2. Inbound vs. Outbound Trade (Real SWIFT Data)
    client_swift = ds['SWIFT'][ds['SWIFT']['entity_name'] == company].copy()
    
    if not client_swift.empty:
        if client_swift['value_zar'].dtype == object:
            client_swift['value_zar'] = client_swift['value_zar'].astype(str).str.replace(',', '').astype(float)
            
        swift_mix = client_swift.groupby('direction')['value_zar'].sum().reset_index()
        swift_mix['value_zar'] /= 1e9 # Scale to Billions (ZARbn)
    else:
        swift_mix = pd.DataFrame({'direction': ['No Data'], 'value_zar': [1.0]})

    fig_donut = px.pie(
        swift_mix, 
        names='direction', 
        values='value_zar', 
        hole=0.6,
        title=f"{company}: Trade Direction & FX Exposure",
        color_discrete_sequence=[outbound_color, inbound_color]
    )

    fig_donut.update_layout(
        plot_bgcolor=bg_color,
        paper_bgcolor=bg_color,
        font=dict(color=text_color),
        annotations=[dict(text='Trade Mix', x=0.5, y=0.5, font_size=20, showarrow=False, font_color=text_color)]
    )

    # Save inside the company's specific subfolder
    fig_donut.write_html(f"{client_dir}/trade_mix.html", include_plotlyjs='cdn')

print(f"All client folders and drill-down assets successfully created under '{base_output_dir}/'!")

Generating organized drill-down folders and plots for 20 companies...
All client folders and drill-down assets successfully created under 'pipeline/dashboard_assets/clients/'!


## JSON Creation for Dashboard and GenAI Interpreter

In [13]:
output_dir = "pipeline"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

sector_multipliers_meta = {
    "Mining":        {"alpha": 0.80, "beta": 1.00, "gamma": 0.40, "delta": 0.10, "epsilon": 1.00},
    "Retail":        {"alpha": 1.00, "beta": 0.50, "gamma": 0.15, "delta": 0.02, "epsilon": 0.30},
    "Telecoms":      {"alpha": 0.90, "beta": 0.80, "gamma": 0.20, "delta": 0.05, "epsilon": 0.90},
    "Industrials":   {"alpha": 0.85, "beta": 0.70, "gamma": 0.25, "delta": 0.06, "epsilon": 0.80},
    "Financials":    {"alpha": 0.40, "beta": 0.60, "gamma": 0.05, "delta": 0.01, "epsilon": 0.20},
    "Healthcare":    {"alpha": 0.90, "beta": 0.85, "gamma": 0.30, "delta": 0.07, "epsilon": 0.75},
    "Real Estate":   {"alpha": 0.50, "beta": 0.20, "gamma": 0.05, "delta": 0.01, "epsilon": 0.85}
}

# Clenaing Data
if 'Competitor_Leakage_ZARbn' in sow_df.columns:
    sow_df = sow_df.sort_values('Competitor_Leakage_ZARbn', ascending=False).reset_index(drop=True)
else:
    sow_df['Competitor_Leakage_ZARbn'] = sow_df['Total_Estimated_Wallet_ZARbn'] - sow_df['Total Capture (ZARbn)']
    sow_df = sow_df.sort_values('Competitor_Leakage_ZARbn', ascending=False).reset_index(drop=True)

sow_df['opportunity_rank'] = sow_df.index + 1
export_df = sow_df.fillna(0)

# Drop any accidental rows where entity_name is missing or numeric
export_df = export_df[export_df['entity_name'].notna()]
export_df['entity_name'] = export_df['entity_name'].astype(str)

clients_data = json.loads(export_df.to_json(orient="records"))

portfolio_summary = {
    "total_portfolio_estimated_wallet_zar_bn": float(export_df['Total_Estimated_Wallet_ZARbn'].sum()),
    "total_portfolio_captured_zar_bn": float(export_df['Total Capture (ZARbn)'].sum()),
    "total_portfolio_competitor_leakage_zar_bn": float(export_df['Competitor_Leakage_ZARbn'].sum()),
    "average_portfolio_share_of_wallet_pct": float(export_df['Share_of_Wallet_%'].mean()),
    "total_entities_analyzed": int(len(export_df))
}

# Map Visualization File Paths for Frontend UI components
plot_assets = {
    "portfolio_level": {
        "wallet_gap_chart": "./dashboard_assets/wallet_gap_chart.html",
        "growth_matrix": "./dashboard_assets/growth_matrix.html",
        "opportunity_heatmap": "./dashboard_assets/opportunity_heatmap.html"
    },
    "client_drill_downs": {}
}

# Dynamically map individual client profile paths safely with string casting
for company in export_df['entity_name']:
    safe_name = str(company).replace(" ", "_").replace(",", "").replace("/", "-")
    plot_assets["client_drill_downs"][company] = {
        "profile_subplot": f"./dashboard_assets/clients/{safe_name}_profile.html",
        "cash_cycle": f"./dashboard_assets_clients/{safe_name}/cash_cycle.html",
        "trade_mix": f"./dashboard_assets_clients/{safe_name}/trade_mix.html"
    }

master_manifest = {
    "metadata": {
        "engine": "Syn Bank Share of Wallet Intelligence Engine",
        "version": "1.0.0",
        "target_audience": ["GenAI Agent Context", "Frontend Dashboard API"],
        "generated_timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    },
    "portfolio_summary": portfolio_summary,
    "methodology_parameters": {
        "sector_multipliers": sector_multipliers_meta,
        "assumptions": {
            "financial_institutions_cogs_proxy": "70% of reported revenue used as operational cost proxy due to lack of standard COGS reporting.",
            "leakage_definition": "Total Estimated Wallet minus Syn Bank Captured Wallet."
        }
    },
    "client_rankings_and_data": clients_data,
    "visualization_manifest": plot_assets
}

# Write to pipeline/results.json
json_path = "pipeline/results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(master_manifest, f, indent=4, default=str)

print(f"Successfully compiled and exported all data, rankings, and plot maps to '{json_path}'!")

Successfully compiled and exported all data, rankings, and plot maps to 'pipeline/results.json'!
